# Agent training

In [ ]:
from training_utils import train_model, further_train_model, evaluate_model

def compare_all_models(algo_dict):
    """
    algo_dict = {
        "RANDOM": None,
        "DQN": "models/v0.7.1_circle_DQN_2024-12-10_13-42-17.zip"
    }
    """
    for algo, model in algo_dict.items():
        evaluate_model(
            algorithm=algo,
            version="v0.7.1",
            env_version="shaping",
            map="circle",
            n_vehicles=5,
            n_episodes= 5,
            render_mode=None,
            model_load_path=model,
            random_seed=123
        )

def make_model_path(model_dir, model_id=None): 
    if model_id is None:
        model_id = model_dir
    return f"models/{model_dir}/{model_id}.zip"

In [ ]:
algo_dict = {
    "RANDOM": None,
    "GREEDY": None,
    "A2C": make_model_path("2025-02-24_17-29-59_v0.6.3_noTime_circle_A2C"),
    # "A2C": make_model_path("2025-02-24_17-20-40_v0.6.3_basic_circle_A2C"),
    # "A2C": make_model_path("2025-02-26_17-50-47_v0.7.1_shaping_circle_A2C"),
}

compare_all_models(algo_dict)

In [ ]:
evaluate_model(algorithm="GREEDY", version="v0.7.1", env_version="noTime", map="circle", n_vehicles=5, n_episodes=5, render_mode=None, model_load_path=None, random_seed=123)

In [ ]:
evaluate_model(
    algorithm="A2C",
    version="v0.7.1",
    env_version="shaping",
    map="circle",
    n_vehicles=5,
    n_episodes= 5,
    render_mode=None,
    model_load_path=make_model_path("2025-02-26_17-50-47_v0.7.1_shaping_circle_A2C"),
    random_seed=123
)

In [ ]:
# Train new model
model_path = train_model("A2C", "MultiInputPolicy", "v0.7.1", "shaping", "circle", n_vehicles=5, n_steps=10000, execution_context="local", random_seed=None)

In [ ]:
# manually configure model load path
model_dir = "v0.7.1_circle_DQN_2024-12-10_13-42-17"
model_id = "v0.7.1_circle_DQN_2024-12-10_13-42-17"
model_path = make_model_path(model_dir, model_id)

In [ ]:
# Train saved model further
model_path = further_train_model("DQN", "v0.7.1", "noTime", "circle", n_vehicles=5, n_steps=20000, model_load_path=model_path)

In [ ]:
# Quick evaluation
# model_path = "models/v0.3_circle_a2c_2024-09-18_23-58-11/v0.3_circle_a2c_2024-09-18_23-58-11.zip"
evaluate_model("DQN", "v0.7.1", "noTime", "circle", n_vehicles=5, n_episodes=5, model_load_path=model_path)

In [ ]:
from stable_baselines3 import PPO, A2C, DQN
from circle_environment import CircleEnv

# Observe execution of trained agent in GUI
env = CircleEnv(render_mode="human", vehicles_to_spawn=5)
model = A2C.load(model_save_name, env)

num_steps = 5
observation, info = env.reset()
for t in range(num_steps):
        actions, _ = model.predict(observation, state=None, deterministic=False)
        observation, reward, terminated, truncated, info = env.step(actions)

env.close()

In [ ]:
# Random actions to compare with the agent
env = CircleEnv(render_mode="human", vehicles_to_spawn=5)
observation, info = env.reset()
try:
    for _ in range(5):
        action = env.action_space.sample() # select a random action
        observation, reward, terminated, truncated, info = env.step(action)
        # if terminated or truncated:
            # observation, info = env.reset()
finally:        
    env.close()

In [ ]:
# --- Benchmark environment ---

def benchmark_env(random_seed):
    env = CircleEnv(render_mode=None, vehicles_to_spawn=15)
    # set seed for reproducability
    import random
    random.seed(random_seed) # needed for batteries of simulation class TODO: make this seedable via the env.seed of gymnasium
    observation, info = env.reset(seed=random_seed)
    env.action_space.seed(random_seed)
    try:
        for _ in range(200):
            action = env.action_space.sample() # select a random action
            observation, reward, terminated, truncated, info = env.step(action)
            if terminated or truncated:
                observation, info = env.reset()
    finally:
        env.close()
    
import cProfile
# cProfile.run('train_model("A2C", "MultiInputPolicy", "v0.3", "circle", n_vehicles=5, n_steps=10000)', 'output.pstats')
benchmark_file = 'output_v0.3.1.pstats'
cProfile.run('benchmark_env(random_seed=1)', benchmark_file)

In [ ]:
import pstats
from pstats import SortKey
p = pstats.Stats(benchmark_file)
p.sort_stats(SortKey.CUMULATIVE).print_stats()